# Subject 0 cache (d2)

Builds the d2 cache from subject 0 training, validation and holdout recordings. The MEG is filtered, resampled to 50 Hz and scaled before saving. The cache also contains word and clip metadata and T5-large layer-12 word embeddings. Both training notebooks use this cache.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(".")
SCRATCH = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")) / "pnpl_d2"
DOWNLOAD_DIR = SCRATCH / "lb100"
HOLDOUT_DIR = SCRATCH / "holdout"
CACHE_DIR = OUTPUT_DIR / "cache"
T5_CACHE_DIR = None
CACHE_VERSION = "d2"
CORPORA = ["sherlock", "timit", "mocha", "podcasts"]
FREQ = 50.0
LOWPASS = 40.0
T5_MODEL = "t5-large"
T5_LAYER = 12
MIN_FREE_GB = 3.0

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pnpl==0.2.0",
        "mne",
        "h5py",
        "transformers",
    ],
    check=True,
)

In [ ]:
import os
import shutil
import time
from collections import Counter
import h5py
import mne
import numpy as np
import pandas as pd
from pnpl.competition import LibriBrainCompetitionHoldout, load_vocabulary
from pnpl.datasets import LibriBrain100
from pnpl.datasets.libribrain100 import RUN_RECORDS
from pnpl.tasks.libribrain import WordClassification

if T5_CACHE_DIR is not None:
    T5_CACHE_DIR = Path(T5_CACHE_DIR)
    for filename in (f"meta_{CACHE_VERSION}.npz", f"t5_{CACHE_VERSION}.bin"):
        path = T5_CACHE_DIR / filename
        if not path.is_file():
            raise FileNotFoundError(f"Missing T5 input: {path}")

mne.set_log_level("ERROR")
for path in (DOWNLOAD_DIR, HOLDOUT_DIR, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient

    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
    print("HF_TOKEN found")
except Exception:
    print("no HF_TOKEN (anonymous downloads, slower)")
vocabulary = load_vocabulary("primary")


def normalise_word(w):
    w = str(w).replace("’", "'").lower()
    return "".join((c for c in w if c.isalnum() or c in "-'"))


vocab_index = {normalise_word(w): i for i, w in enumerate(vocabulary)}
assert len(vocab_index) == 50, "vocabulary collides under normalisation"
records = [
    r
    for r in RUN_RECORDS
    if r.subject == "0"
    and r.corpus in CORPORA
    and (r.partition in ("train", "validation"))
]
print(f"{len(records)} runs:", dict(Counter((r.corpus for r in records))))

In [ ]:
def filter_and_resample(x250):
    x = np.asarray(x250, dtype=np.float64)
    x = mne.filter.filter_data(x, 250.0, None, LOWPASS, n_jobs=-1, verbose=False)
    x = mne.filter.resample(x, up=FREQ, down=250.0, axis=-1, npad="auto", verbose=False)
    return x.T


def iqr_scale(x):
    q25, q75 = np.percentile(x, [25, 75], axis=0)
    scale = np.where(q75 - q25 > 0, q75 - q25, 1.0)
    return ((x / scale).astype(np.float16), scale.astype(np.float32))


def open_run(run_key):
    return LibriBrain100(
        data_path=str(DOWNLOAD_DIR),
        task=WordClassification(tmin=0.0, tmax=1.0),
        include_run_keys=[run_key],
        standardize=False,
        clipping_boundary=None,
        download=True,
    )


def read_words(ds, run_key):
    events_path = ds.ensure_file(ds.get_events_path(*run_key))
    events = pd.read_csv(events_path, sep="\t")
    events = events[events["kind"] == "word"].dropna(subset=["segment", "timemeg"])
    if "wavile" in events.columns:
        stimuli = events["wavile"].astype(str)
    else:
        stimuli = pd.Series([""] * len(events))
    if "sentenceidx" in events.columns:
        sentences = events["sentenceidx"]
    else:
        sentences = pd.Series([-1] * len(events))
    onsets = events["timemeg"].to_numpy(np.float64)
    words = events["segment"].astype(str).str.strip().to_numpy()
    sentences = np.nan_to_num(sentences.to_numpy(np.float64), nan=-1).astype(np.int64)
    return onsets, words, sentences, stimuli.to_numpy()


def channel_names(h5_path):
    with h5py.File(h5_path, "r") as f:
        a = f.attrs["channel_names"]
    if isinstance(a, bytes):
        a = a.decode()
    if isinstance(a, str):
        return [s.strip() for s in a.split(",")]
    return [s.decode() if isinstance(s, bytes) else str(s) for s in a]


def filter_attrs(h5_path):
    with h5py.File(h5_path, "r") as f:
        return (
            float(f.attrs.get("highpass_cutoff", np.nan)),
            float(f.attrs.get("lowpass_cutoff", np.nan)),
        )


def remove_download(path):
    try:
        os.remove(path)
    except FileNotFoundError:
        pass
    LibriBrain100._download_futures.pop(path, None)


def free_gb(p):
    return shutil.disk_usage(p).free / 2**30

In [ ]:
t0 = time.time()
meg = open(CACHE_DIR / f"meg_{CACHE_VERSION}.bin", "wb")
word_ids = {w: i for i, w in enumerate(sorted(vocab_index))}
stimulus_ids = {"": 0}
rows, runs, offset, fetched = ([], [], 0, 0.0)
ch_names, checked_filters = (None, False)


def get_id(table, key):
    if key not in table:
        table[key] = len(table)
    return table[key]


for k, r in enumerate(records, 1):
    for label, path in (("downloads", DOWNLOAD_DIR), ("cache", CACHE_DIR)):
        assert (
            free_gb(path) >= MIN_FREE_GB
        ), f"only {free_gb(path):.1f} GB free in {label}"
    ds = open_run(r.run_key)
    assert float(ds.sfreq) == 250.0, f"h5 is {ds.sfreq} Hz, expected 250"
    h5 = ds.get_h5_path(*r.run_key)
    fetched += os.path.getsize(h5) / 2**30
    if not checked_filters:
        hp, lp = filter_attrs(h5)
        print(f"  h5 filter attrs: highpass {hp} Hz, lowpass {lp} Hz")
        assert (
            not hp > 0.1 + 1e-06
        ), f"h5 highpass is {hp} Hz, above the 0.1 Hz we assume"
        assert not lp < LOWPASS, f"h5 lowpass is {lp} Hz, below our {LOWPASS} Hz target"
        ch_names = channel_names(h5)
        checked_filters = True
    x, scale = iqr_scale(filter_and_resample(ds.get_h5_dataset(r.run_key)[:]))
    meg.write(x.tobytes())
    onsets, word_text, sents, stim_text = read_words(ds, r.run_key)
    a = offset + np.rint(onsets * FREQ).astype(np.int64)
    keep = (a >= offset) & (a < offset + len(x))
    run_id = len(runs)
    for i in np.flatnonzero(keep):
        rows.append(
            (
                a[i],
                get_id(word_ids, normalise_word(word_text[i])),
                sents[i],
                get_id(stimulus_ids, str(stim_text[i])),
                run_id,
            )
        )
    runs.append(
        (
            "/".join(r.run_key),
            r.session,
            r.task,
            r.corpus,
            r.partition,
            offset,
            len(x),
            scale,
        )
    )
    offset += len(x)
    ds.close_h5_files()
    del ds, x
    remove_download(h5)
    print(
        f"  [{k:3d}/{len(records)}] {'/'.join(r.run_key):26s} {offset / FREQ / 3600:5.1f} h | {len(rows):7d} words | fetched {fetched:5.1f} GB | cache {meg.tell() / 2 ** 30:5.2f} GB | free {free_gb(DOWNLOAD_DIR):5.1f}/{free_gb(CACHE_DIR):5.1f} GB",
        flush=True,
    )
meg.close()
print(
    f"\n{offset / FREQ / 3600:.1f} h at {FREQ:g} Hz | {len(rows)} word events | {fetched:.1f} GB downloaded, none kept | {(time.time() - t0) / 60:.0f} min"
)

In [ ]:
hold = LibriBrainCompetitionHoldout(track="deep", data_path=str(HOLDOUT_DIR))
hmeta = pd.DataFrame(hold.metadata)
print(hold.counts())
sentence_clips = np.load(hold._ensure_file(0, "sentence"))
word_clips = np.load(hold._ensure_file(0, "word"))
assert float(sentence_clips["sfreq"]) == float(word_clips["sfreq"]) == 250.0, (
    sentence_clips["sfreq"],
    word_clips["sfreq"],
)
n_times = sentence_clips["sentence_n_times"].astype(np.int64)
mask = np.asarray(sentence_clips["sentence_sample_mask"])
for i, n in enumerate(n_times):
    assert mask[i, :n].all() and (
        not mask[i, n:].any()
    ), f"sentence {i}: valid samples are not a prefix, trimming by n_times would be wrong"
epochs = [
    filter_and_resample(sentence_clips["meg"][i, :, : n_times[i]])
    for i in range(len(n_times))
]
n_sentence = len(epochs)
epochs += [
    filter_and_resample(word_clips["meg"][i]) for i in range(len(word_clips["meg"]))
]
flat, hold_scale = iqr_scale(np.concatenate(epochs, axis=0))
flat.tofile(CACHE_DIR / f"hold_{CACHE_VERSION}.bin")
assert np.isfinite(flat).all(), "non-finite holdout samples"
hold_len = np.array([len(e) for e in epochs], dtype=np.int64)
hold_start = np.concatenate([[0], np.cumsum(hold_len)[:-1]])
is_word = hmeta["source"].to_numpy() == "word"
hold_epoch = np.where(
    is_word, n_sentence + hmeta["epoch"].to_numpy(), hmeta["epoch"].to_numpy()
).astype(np.int64)
hold_onset = hmeta["onset_s"].to_numpy(np.float32)
hold_source = is_word.astype(np.int8)
hold_at = hold_start[hold_epoch] + np.rint(hold_onset * FREQ).astype(np.int64)
order_ok = all(
    (np.all(np.diff(hold_onset[hold_epoch == e]) >= 0) for e in np.unique(hold_epoch))
)
assert order_ok, "rows within a holdout epoch are not in onset order"
room = hold_start[hold_epoch] + hold_len[hold_epoch] - hold_at
print(
    f"{len(epochs)} epochs ({n_sentence} sentence + {len(epochs) - n_sentence} word) | {len(hmeta)} rows | {flat.nbytes / 2 ** 20:.0f} MB at {FREQ:g} Hz"
)
for d in (1.0, 1.5, 2.0, 3.0):
    print(f"  {d:.1f} s servable for {(room >= d * FREQ).mean():6.1%} of rows")

In [ ]:
layout = mne.channels.read_layout("Vectorview-all")
key = {n.replace(" ", "").upper(): i for i, n in enumerate(layout.names)}
assert ch_names is not None, "no run was read, so there are no channel names"
idx = [key[n.replace(" ", "").upper()] for n in ch_names]
pos = layout.pos[idx, :2].astype(np.float32)
pos = (pos - pos.min(0)) / (pos.max(0) - pos.min(0))
print(
    f"channel positions {pos.shape} in [{pos.min():.2f}, {pos.max():.2f}], matched by name"
)
words = [w for w, _ in sorted(word_ids.items(), key=lambda kv: kv[1])]

if T5_CACHE_DIR is None:
    import torch
    from transformers import AutoTokenizer, T5EncoderModel

    tok = AutoTokenizer.from_pretrained(T5_MODEL)
    t5 = T5EncoderModel.from_pretrained(T5_MODEL).eval()
    emb = np.zeros((len(words), t5.config.d_model), dtype=np.float32)
    with torch.inference_mode():
        for a in range(0, len(words), 128):
            chunk = words[a : a + 128]
            enc = tok(
                chunk, return_tensors="pt", padding=True, add_special_tokens=False
            )
            hs = t5(**enc, output_hidden_states=True).hidden_states
            m = enc["attention_mask"][..., None]
            emb[a : a + 128] = ((hs[T5_LAYER] * m).sum(1) / m.sum(1)).numpy()
            if a % 4096 == 0:
                print(f"  t5 {a}/{len(words)}", flush=True)
    assert (
        len(hs) == 25 and T5_LAYER == 25 // 2
    ), f"{len(hs)} hidden states, layer {T5_LAYER}"
    emb.astype(np.float16).tofile(CACHE_DIR / f"t5_{CACHE_VERSION}.bin")
    print(f"t5 embeddings {emb.shape} from layer {T5_LAYER} of {len(hs)}")
else:
    with np.load(
        T5_CACHE_DIR / f"meta_{CACHE_VERSION}.npz", allow_pickle=False
    ) as t5_metadata:
        assert np.array_equal(
            t5_metadata["words"], np.asarray(words)
        ), "T5 word order differs."
        assert np.array_equal(
            t5_metadata["vocab"], np.asarray(vocabulary)
        ), "T5 vocabulary differs."
        t5_dimension = int(t5_metadata["t5_dim"])
    source_t5 = T5_CACHE_DIR / f"t5_{CACHE_VERSION}.bin"
    assert (
        source_t5.stat().st_size == len(words) * t5_dimension * 2
    ), "Unexpected T5 file size."
    emb = np.memmap(source_t5, dtype=np.float16, mode="r").reshape(
        len(words), t5_dimension
    )
    shutil.copyfile(source_t5, CACHE_DIR / source_t5.name)
    print(f"T5 embeddings {emb.shape} loaded from {source_t5}")

In [ ]:
r = np.array(rows, dtype=np.int64)
stims = [s for s, _ in sorted(stimulus_ids.items(), key=lambda kv: kv[1])]
label = np.array([vocab_index.get(words[i], -1) for i in r[:, 1]], dtype=np.int16)
np.savez(
    CACHE_DIR / f"meta_{CACHE_VERSION}.npz",
    version=CACHE_VERSION,
    freq=FREQ,
    lowpass=LOWPASS,
    n_samples=offset,
    vocab=np.array(vocabulary),
    words=np.array(words),
    stims=np.array(stims),
    t5_dim=emb.shape[1],
    positions=pos,
    onset=r[:, 0],
    word_id=r[:, 1].astype(np.int32),
    label=label,
    sentence=r[:, 2],
    stim_id=r[:, 3].astype(np.int32),
    run=r[:, 4].astype(np.int32),
    run_name=np.array([a[0] for a in runs]),
    run_session=np.array([a[1] for a in runs]),
    run_task=np.array([a[2] for a in runs]),
    run_corpus=np.array([a[3] for a in runs]),
    run_partition=np.array([a[4] for a in runs]),
    run_offset=np.array([a[5] for a in runs], dtype=np.int64),
    run_len=np.array([a[6] for a in runs], dtype=np.int64),
    run_scale=np.stack([a[7] for a in runs]),
    hold_epoch=hold_epoch,
    hold_onset=hold_onset,
    hold_at=hold_at,
    hold_source=hold_source,
    hold_start=hold_start,
    hold_len=hold_len,
    hold_scale=hold_scale,
    hold_indices=np.asarray(hold.indices),
)
counts = np.bincount(label[label >= 0], minlength=50)
missing = [vocabulary[c] for c in range(50) if counts[c] == 0]
assert not missing, f"vocabulary words absent from the events: {missing}"
print(
    f"in-vocabulary words: {(label >= 0).sum()} of {len(label)} ({(label >= 0).mean():.0%})"
)
print(
    f"sentence ids present for {(r[:, 2] >= 0).mean():.0%} of words ({len(np.unique(r[:, 2]))} distinct values per recording)"
)
print(
    f"stimulus ids present for {(r[:, 3] > 0).mean():.0%} of words ({len(stims) - 1} distinct)"
)
print("\nrarest vocabulary words:")
for c in np.argsort(counts)[:6]:
    print(f"  {vocabulary[c]:>8s} {counts[c]:6d}")
for f in sorted(CACHE_DIR.glob(f"*_{CACHE_VERSION}.*")):
    print(f"\n  {f.name:20s} {f.stat().st_size / 2 ** 30:6.2f} GB")
shutil.rmtree(HOLDOUT_DIR, ignore_errors=True)